# Step 2 — 카테고리별 인기 태그 추출

`step1_tags_parsed.csv` 로드 → 타깃 3개 카테고리(Gaming / Music / Film & Animation) 내 태그 document-frequency 집계.

- 영상당 set으로 중복 제거 (= 몇 개 영상에 등장했나)
- 불용어 제거: `the`
- 카테고리 내 빈도 ≥5 필터 → 상위 15 추출
- 카테고리 간 겹침/고유 태그 분석
- 산출: `step2_top_tags.csv` (category·tag·df_count; Step 3 입력)

In [6]:
import pandas as pd
from collections import Counter

In [7]:
d = pd.read_csv('step1_tags_parsed.csv')
d['tag_list'] = d['tags_joined'].fillna('').apply(lambda s: s.split('|') if s else [])

TARGET = ['Gaming', 'Music', 'Film & Animation']
STOPLIST = {'the'}   # 불용어 제거 (발표용 정리)
MIN_DF = 5      # 카테고리 내 document-frequency 임계
TOP_N = 15
print('loaded:', d.shape)

loaded: (6249, 5)


In [8]:
# 카테고리별 태그 document-frequency (영상당 set 중복 제거)
top_tags = {}
for cat in TARGET:
    sub = d[d['category'] == cat]
    cnt = Counter()
    for tags in sub['tag_list']:
        cnt.update(set(tags))
    filtered = [(t, c) for t, c in cnt.most_common() if c >= MIN_DF and t not in STOPLIST]
    top_tags[cat] = filtered[:TOP_N]
    print('===', cat, '(n_videos=' + str(len(sub)) + ', 임계>=' + str(MIN_DF) + ' 통과=' + str(len(filtered)) + ') ===')
    for t, c in top_tags[cat]:
        print('  {:3d}  {}'.format(c, t))
    print()

=== Gaming (n_videos=101, 임계>=5 통과=32) ===
   24  gameplay
   22  nintendo
   18  nintendo switch
   17  video game
   17  action
   17  game
   15  rpg
   15  play
   14  fun
   13  kids
   12  play nintendo
   12  adventure
    9  xbox one
    8  games
    8  e3 2018

=== Music (n_videos=792, 임계>=5 통과=337) ===
  166  pop
   93  music
   76  records
   54  music video
   50  official
   49  alternative
   46  live
   39  country
   38  official video
   33  lyrics
   33  rap
   31  rca records label
   30  hip hop
   27  song
   27  island

=== Film & Animation (n_videos=306, 임계>=5 통과=130) ===
   74  trailer
   47  movie
   43  animation
   36  film
   25  comedy
   25  honest trailers
   25  honest trailer
   24  official
   23  official trailer
   23  screenjunkies
   23  screen junkies
   21  disney
   20  2018
   18  animated
   17  movies



In [9]:
# Top 태그 겹침 / 고유
sets = {cat: set(t for t, _ in top_tags[cat]) for cat in TARGET}
for i, a in enumerate(TARGET):
    for b in TARGET[i+1:]:
        common = sorted(sets[a] & sets[b])
        print(a, '∩', b, ':', common if common else '(없음)')
all3 = sorted(sets[TARGET[0]] & sets[TARGET[1]] & sets[TARGET[2]])
print('3개 공통:', all3 if all3 else '(없음)')

Gaming ∩ Music : (없음)
Gaming ∩ Film & Animation : (없음)
Music ∩ Film & Animation : ['official']
3개 공통: (없음)


In [10]:
# Step 3 입력용 저장
rows = []
for cat in TARGET:
    for t, c in top_tags[cat]:
        rows.append({'category': cat, 'tag': t, 'df_count': c})
top_df = pd.DataFrame(rows)
top_df.to_csv('step2_top_tags.csv', index=False)
print('saved: step2_top_tags.csv', top_df.shape)

saved: step2_top_tags.csv (45, 3)
